# TASK:

use encapsulation for class i:e., StudentLogin():
initialize two static but private attributes i:e., __user_name, __password
create one private methods i:e., name __dbconectivity() that will validate you username & password. if both are valid then should print "valid user" otherwise "not valid"
create one public methods i:e., update password & it should update your private password

In [ ]:

class StudentLogin:

    def __init__(self):
        self.__user_name = "Ahad"
        self.__password = "casetoo123"

    # Private Method
    def __dbconectivity(self, username, password):

        if username == self.__user_name and password == self.__password:
            print("valid user")
        else:
            print("not valid")

    # Public Method
    def update_password(self, new_password):
        self.__password = new_password
        print("Password updated successfully")


# Object Creation
student = StudentLogin()

# Login Validation
student._StudentLogin__dbconectivity("Ahad", "casetoo123")

# Update Password
student.update_password("786786")
 
# Login Again with Updated Password
student._StudentLogin__dbconectivity("Ahad", "786786")


valid user
Password updated successfully
valid user


# Task 01 |

. Nested Transaction Analyzer
. Concepts: Data structures, sorting, comprehensions


In [5]:
from collections import defaultdict


def analyze_transactions(transactions):
    seen, spending, cats, customers = (
        set(),
        defaultdict(float),
        defaultdict(lambda: defaultdict(float)),
        set(),
    )

    for t in transactions:
        customers.add(t["customer"])
        if t["id"] in seen:
            continue
        seen.add(t["id"])  # Duplicate ID skip

        if t["status"] == "success":  # Sirf successful transactions
            spending[t["customer"]] += t["amount"]
            cats[t["customer"]][t["category"]] += t["amount"]

    # Report banana aur top category nikalna
    report = [
        {
            "customer": c,
            "successful_spending": spending[c],
            "top_category": (
                min(cats[c].items(), key=lambda x: (-x[1], x[0]))[0]
                if cats[c]
                else None
            ),
        }
        for c in customers
    ]

    # Sort karna (Spending descending, Naam ascending) aur Rank dena
    report.sort(key=lambda x: (-x["successful_spending"], x["customer"]))
    for i, r in enumerate(report, 1):
        r["rank"] = i

    return report

# Task 02 |

 . Longest Valid Window
. Concepts: Sliding window, dictionaries
 Problem: Write longest_unique_window(text) to find the longest contiguous substring with no repeated characters. Return the ..  substring, start index, and length.
. Requirements: Target O(n) time. On equal-length windows, select the earliest. Treat spaces and punctuation as characters.
Sample / expected behavior: Input: "abcabcbb" -> ("abc", 0, 3); "" -> ("", 0, 0).


In [6]:
def longest_unique_window(text):
    char_index = {}  # Character ka last seen index store karne ke liye dictionary
    start = 0  # Sliding window ka starting index
    max_len = 0  # Sab se lambi unique window ki length
    best_start = 0  # Sab se lambi window ka real starting point

    for end, char in enumerate(text):
        # Agar character pehle aa chuka hai aur current window ke andar hai
        if char in char_index and char_index[char] >= start:
            start = (
                char_index[char] + 1
            )  # Window ka start aage shift kar do taaki duplicate khatam ho

        char_index[char] = end  # Character ka naya index save karo

        # Agar current window purani max length se bari hai
        # (Strict '>' lagane se equal length mein pehli wali hi rehti hai)
        if (end - start + 1) > max_len:
            max_len = end - start + 1
            best_start = start

    # Result return karo: (Substring, Start Index, Length)
    return (text[best_start : best_start + max_len], best_start, max_len)


# --- Sample Tests ---
print(longest_unique_window("abcabcbb"))  # Output: ('abc', 0, 3)
print(longest_unique_window(""))  # Output: ('', 0, 0)

('abc', 0, 3)
('', 0, 0)


# Task 03 |

 . Recursive JSON Path Explorer
. Concepts: Recursion, nested lists/dictionaries
. Problem: Implement find_paths(data, target) that recursively searches nested dictionaries and lists and returns all paths whose value equals target.
. Requirements: Use dot notation for dictionary keys and [index] for list positions. Preserve traversal order; do not treat bool True as numeric 1.
. Sample / expected behavior: Input: {"a":[{"x":5},5],"b":{"y":5}}, target=5 -> ["a[0].x", "a[1]", "b.y"].


In [7]:
def find_paths(data, target):
    paths = []

    # Helper function jo recursively data ke andar jaye ga
    def helper(current, path):
        # Agar value target se match kare (aur data type bhi same ho, taake True == 1 na ho)
        if current == target and type(current) is type(target):
            paths.append(path)

        # Agar current value ek Dictionary hai
        if isinstance(current, dict):
            for k, v in current.items():
                new_path = f"{path}.{k}" if path else k
                helper(v, new_path)

        # Agar current value ek List hai
        elif isinstance(current, list):
            for i, v in enumerate(current):
                new_path = f"{path}[{i}]"
                helper(v, new_path)

    helper(data, "")
    return paths


# --- Sample Test ---
data = {"a": [{"x": 5}, 5], "b": {"y": 5}}
print(find_paths(data, 5))
# Output: ['a[0].x', 'a[1]', 'b.y']

['a[0].x', 'a[1]', 'b.y']


 # Task 04 |
.  Dependency Execution Planner
. Concepts: Graphs, sets, algorithms
. Problem: Given tasks and prerequisite pairs (A,B) meaning A must finish before B, return a valid execution order.
. Requirements: Detect cycles and raise ValueError with a clear message. For multiple available tasks, select alphabetically first. Reject unknown task names.
. Sample / expected behavior: Tasks=["build","test","deploy"]; dependencies=[("build","test"),("test","deploy")] -> ["build","test","deploy"].


In [8]:
import heapq


def get_execution_order(tasks, dependencies):
    t_set = set(tasks)
    graph, in_degree = {t: [] for t in tasks}, {t: 0 for t in tasks}

    # Step 1: Graph aur in-degree banana (+ unknown check)
    for a, b in dependencies:
        if a not in t_set or b not in t_set:
            raise ValueError("Unknown task")
        graph[a].append(b)
        in_degree[b] += 1

    # Step 2: 0-dependency tasks ko Min-Heap mein daalna (alphabetical order ke liye)
    heap = [t for t in tasks if in_degree[t] == 0]
    heapq.heapify(heap)
    order = []

    # Step 3: Tasks ko process karna
    while heap:
        curr = heapq.heappop(heap)
        order.append(curr)
        for n in graph[curr]:
            in_degree[n] -= 1
            if in_degree[n] == 0:
                heapq.heappush(heap, n)

    # Step 4: Agar saare tasks cover na hon toh cycle detect hui hai
    if len(order) != len(tasks):
        raise ValueError("Cycle detected!")

    return order

# Task 05 |
. Expression Validator and Evaluator
. Concepts: Stacks, parsing, exception handling
. Problem: Create a calculator for integer expressions containing +, -, *, /, parentheses, and whitespace. Apply standard operator .precedence and parentheses.
. Requirements: Do not use eval(), exec(), or third-party parsers. Handle unary minus, malformed expressions, and division by zero.
Sample / expected behavior: Input: "-(3 + 2) * 4 + 18 / 3" -> -14.0.


In [ ]:
import re


def evaluate_expression(expr):
    # Step 1: Numbers aur operators alag karna
    tokens = re.findall(r"\d+|[+\-*/()]", expr)
    vals, ops = [], []
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}

    def calc():
        op, b, a = ops.pop(), vals.pop(), vals.pop()
        vals.append(
            a + b
            if op == "+"
            else a - b
            if op == "-"
            else a * b
            if op == "*"
            else a / b
        )

    i = 0
    while i < len(tokens):
        t = tokens[i]
        if t.isdigit():
            vals.append(float(t))  # Number hai toh values mein dalo
        elif t == "(":
            ops.append(t)  # Opening bracket
        elif t == ")":  # Closing bracket aate hi bracket solve karo
            while ops[-1] != "(":
                calc()
            ops.pop()
        elif t in "+-*/":
            # Unary minus handle karne ke liye (jaise -(3))
            if t == "-" and (i == 0 or tokens[i - 1] in "(+-*/"):
                vals.append(0.0)
            while ops and ops[-1] != "(" and prec.get(ops[-1], 0) >= prec[t]:
                calc()
            ops.append(t)
        i += 1

    while ops:
        calc()  # Baaki bache hue operators solve karo
    return vals[0]


# Test
print(evaluate_expression("-(3 + 2) * 4 + 18 / 3"))  # Output: -14.0

# Task 06 |
.  Banking Ledger with OOP
. Concepts: Classes, encapsulation, properties
. Problem: Build Account and Bank classes. Support deposit, withdrawal, transfer, and account statements. Maintain immutable transaction records containing transaction ID, action, amount, and resulting balance.
. Requirements: Reject nonpositive amounts, insufficient funds, and nonexistent accounts. A transfer must be atomic: both balances update or neither does.
. Sample / expected behavior: Account A=1000, B=200; transfer 350 from A to B -> A=650, B=550.


In [9]:
class Account:
    def __init__(self, id, balance=0):
        # Account ki ID, current balance, aur transaction history initialize karna
        self.id, self.bal, self.txs = id, float(balance), []

    def update(self, amt, action):
        # Step 1: Check karo ke amount positive ho aur withdrawal ke liye balance kafi ho
        if amt <= 0 or (action == "WITHDRAW" and amt > self.bal):
            raise ValueError("Invalid amount or low balance")
        
        # Step 2: Deposit ho toh balance barhao, Withdraw ho toh kam karo
        self.bal += amt if action == "DEPOSIT" else -amt
        
        # Step 3: Transaction ka record history mein save karo (immutable record)
        self.txs.append({"action": action, "amount": amt, "balance": self.bal})

class Bank:
    def __init__(self):
        # Bank ke saare accounts store karne ke liye dictionary
        self.accs = {}

    def add(self, id, bal=0):
        # Naya account create karke bank mein add karna
        self.accs[id] = Account(id, bal)

    def transfer(self, f, t, amt):
        # Step 1: Check karo ke dono accounts bank mein mojood hain ya nahi
        if f not in self.accs or t not in self.accs:
            raise ValueError("Account not found")
        
        # Step 2: Atomic Transfer - Pehle sender se paise nikalo, phir receiver ko do
        self.accs[f].update(amt, "WITHDRAW")
        self.accs[t].update(amt, "DEPOSIT")

# --- Sample Test ---
bank = Bank()
bank.add("A", 1000)  # Account A banaya 1000 balance ke sath
bank.add("B", 200)   # Account B banaya 200 balance ke sath

bank.transfer("A", "B", 350)  # A se B mein 350 transfer kiye
print("A Balance:", bank.accs["A"].bal)  # Output: 650.0
print("B Balance:", bank.accs["B"].bal)  # Output: 550.0

A Balance: 650.0
B Balance: 550.0


# Task 07 |

. Employee Payroll Engine
. Concepts: Inheritance, polymorphism, abstract classes
. Problem: Define abstract Employee with calculate_pay(). Implement SalariedEmployee, HourlyEmployee, and CommissionEmployee with different pay formulas. Generate a unified payroll summary.
. Requirements: Hourly overtime above 40 hours pays 1.5x. Commission employees receive base + sales*rate. Prevent direct instantiation of Employee.
. Sample / expected behavior: Hourly rate=20, hours=45 -> 950; Commission base=500, sales=4000, rate=.10 -> 900.


In [10]:

from abc import ABC, abstractmethod


# Abstract Base Class (Yeh direct instantiate nahi ho sakti)
class Employee(ABC):

    @abstractmethod
    def calculate_pay(self):
        pass


# 1. Salaried Employee: Fixed weekly salary milti hai
class SalariedEmployee(Employee):

    def __init__(self, name, salary):
        self.name, self.salary = name, salary

    def calculate_pay(self):
        return self.salary


# 2. Hourly Employee: 40 hours se upar overtime ka 1.5x rate milta hai
class HourlyEmployee(Employee):

    def __init__(self, name, rate, hours):
        self.name, self.rate, self.hours = name, rate, hours

    def calculate_pay(self):
        if self.hours <= 40:
            return self.rate * self.hours
        # 40 hours tak normal rate + extra hours ka 1.5x overtime
        return (40 * self.rate) + ((self.hours - 40) * self.rate * 1.5)


# 3. Commission Employee: Base salary + (Total Sales * Rate)
class CommissionEmployee(Employee):

    def __init__(self, name, base, sales, rate):
        self.name, self.base, self.sales, self.rate = name, base, sales, rate

    def calculate_pay(self):
        return self.base + (self.sales * self.rate)


# --- Sample Test & Summary ---
employees = [
    SalariedEmployee("Ali", 1200),
    HourlyEmployee("Sara", 20, 45),  # 40*20 + 5*30 = 950.0
    CommissionEmployee("Ahmed", 500, 4000, 0.10),  # 500 + 400 = 900.0
]

# Polymorphism: Sab ka calculate_pay() aik hi loop mein chal jata hai
for emp in employees:
    print(f"{emp.name}: {emp.calculate_pay()}")

Ali: 1200
Sara: 950.0
Ahmed: 900.0


# Task 08 |
 . Cache Decorator with Expiration
. Concepts: Decorators, closures, functools, time
. Problem: Implement @ttl_cache(seconds) that caches results for repeated positional and keyword arguments and recomputes after expiration. Track cache hits and misses.
. Requirements: Preserve function metadata with functools.wraps. Handle equivalent keyword argument order; document behavior for unhashable inputs. Use time.monotonic().
. Sample / expected behavior: A call with x=2,y=3 followed by y=3,x=2 before expiry should return a cache hit.


In [11]:
from functools import wraps
import time


def ttl_cache(seconds):
    def decorator(func):
        cache = {}  # Cache store: key -> (result, expiry_time)
        hits = misses = 0  # Cache hits aur misses track karne ke variables

        @wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal hits, misses
            # Keyword arguments ko sort karte hain taaki order change hone par bhi key same rahe
            key = (args, tuple(sorted(kwargs.items())))
            now = time.monotonic()

            # Step 1: Check karo ke key cache mein hai aur expire nahi hui (Cache Hit)
            if key in cache and cache[key][1] > now:
                hits += 1
                return cache[key][0]

            # Step 2: Agar cache mein nahi hai ya expire ho gayi (Cache Miss)
            misses += 1
            res = func(*args, **kwargs)
            # Naya result aur uska expiry time (current time + seconds) save karo
            cache[key] = (res, now + seconds)
            return res

        # Hits aur misses dekhne ke liye functions attach kiye hain
        wrapper.hits = lambda: hits
        wrapper.misses = lambda: misses
        return wrapper

    return decorator


# --- Sample Test ---
@ttl_cache(seconds=2)
def add(x, y):
    print("Computing...")  # Jab calculate hoga tab yeh print hoga
    return x + y


print(add(x=2, y=3))  # Miss (Computation hogi)
print(add(y=3, x=2))  # Hit (Keyword order change hone ke bawajood cache se uthayega!)
print(
    f"Hits: {add.hits()}, Misses: {add.misses()}"
)  # Output: Hits: 1, Misses: 1

Computing...
5
5
Hits: 1, Misses: 1


# Task 09 | 
. Streaming Log Inspector
. Concepts: Generators, iterators, file handling
. Problem: Write a generator read_errors(path) that streams a log file line by line, yielding (line_number, timestamp, message) only for ERROR records. Build summarize_errors() to count errors by date.
. Requirements: Do not load the whole file into memory. Skip malformed lines while tracking their count. Expected format: YYYY-MM-DD HH:MM:SS | LEVEL | message.
. Sample / expected behavior: Input line: 2026-11-07 10:30:00 | ERROR | Connection lost -> (1,"2026-11-07 10:30:00","Connection lost").


In [12]:
from collections import defaultdict


# Generator jo file ko line by line stream karega aur sirf ERROR records nikalega
def read_errors(path):
    malformed_count = 0
    try:
        with open(path, "r") as f:
            for line_no, line in enumerate(f, 1):
                try:
                    # Format: YYYY-MM-DD HH:MM:SS | LEVEL | message ko split karna
                    parts = line.strip().split(" | ")
                    if len(parts) != 3:
                        malformed_count += 1
                        continue

                    timestamp, level, message = parts

                    # Agar log ka level ERROR hai toh yield (return) karo
                    if level == "ERROR":
                        date = timestamp.split()[
                            0
                        ]  # Date alag karna (YYYY-MM-DD)
                        yield line_no, timestamp, message, date
                except Exception:
                    malformed_count += 1  # Ghalat line ko skip karke count barhana
    except FileNotFoundError:
        print("File nahi mili!")


# Errors ko date ke hisaab se count karne ka function
def summarize_errors(path):
    error_counts = defaultdict(int)

    # generator function ko call karke errors count karna
    for line_no, timestamp, message, date in read_errors(path):
        error_counts[date] += 1

    return dict(error_counts)


# --- Sample Test (Dummy file create karke test karna) ---
# Testing ke liye sample log data write kiya
with open("sample.log", "w") as f:
    f.write("2026-11-07 10:30:00 | INFO | Server started\n")
    f.write("2026-11-07 10:31:00 | ERROR | Connection lost\n")
    f.write("2026-11-08 09:15:00 | ERROR | Database timeout\n")
    f.write("Malformed line without proper format\n")

# Summary check karna
print(summarize_errors("sample.log"))
# Output: {'2026-11-07': 1, '2026-11-08': 1}

{'2026-11-07': 1, '2026-11-08': 1}


# Task 10 | 
. CSV Inventory Reconciliation
. Concepts: CSV, file handling, validation
. Problem: Read inventory.csv (sku,name,quantity) and movements.csv (sku,type,units). Apply IN and OUT movements in order, then write updated_inventory.csv and rejected_movements.csv.
. Requirements: Reject unknown SKUs, negative or zero units, invalid movement types, and any OUT causing negative stock. Continue processing after rejected rows.
. Sample / expected behavior: SKU P01 starts at 10; OUT 4, OUT 9, IN 2 -> final 8; second movement rejected.


In [13]:
import csv


def reconcile_inventory(inv_path, mov_path, out_inv, out_rej):
    # Step 1: Inventory read karke dictionary banana
    inv = {
        r["sku"]: {"name": r["name"], "quantity": int(r["quantity"])}
        for r in csv.DictReader(open(inv_path))
    }
    rejected = []

    # Step 2: Movements process karna aur validation check karna
    for r in csv.DictReader(open(mov_path)):
        sku, t, u = r["sku"], r["type"], int(r["units"])

        # Unknown SKU, ghalat type ya zero/negative units check
        if sku not in inv or t not in ("IN", "OUT") or u <= 0:
            rejected.append({**r, "reason": "Invalid data"})
            continue

        # OUT movement mein stock kam hone par reject karna
        if t == "OUT" and inv[sku]["quantity"] < u:
            rejected.append({**r, "reason": "Low stock"})
            continue

        # Stock update karna (IN hai toh add, OUT hai toh minus)
        inv[sku]["quantity"] += u if t == "IN" else -u

    # Step 3: Updated inventory file save karna
    csv.DictWriter(
        open(out_inv, "w", newline=""), ["sku", "name", "quantity"]
    ).writeheader() and [
        csv.DictWriter(
            open(out_inv, "a", newline=""), ["sku", "name", "quantity"]
        ).writerow({"sku": s, **d})
        for s, d in inv.items()
    ]

    # Step 4: Rejected movements file save karna
    writer = csv.DictWriter(
        open(out_rej, "w", newline=""), ["sku", "type", "units", "reason"]
    )
    writer.writeheader()
    writer.writerows(rejected)

# Task 11 |
.  Multi-File Student Gradebook
. Concepts: File handling, dictionaries, exceptions
. Problem: Merge students.csv (student_id,name) and scores.csv (student_id,subject,marks). Calculate per-student average, grade, and rank; export report.csv.
. Requirements: Marks must be 0–100; reject unknown IDs and duplicate student-subject entries. Grades: A>=85, B>=70, C>=55, D>=40, otherwise F. Rank by average, then ID.
. Sample / expected behavior: Student 101 with marks 90,80,85 -> average 85.0, grade A.


In [15]:
import csv
from collections import defaultdict


def generate_gradebook(students_path, scores_path, output_path):
    # Step 1: Students data read karna (ID -> Name)
    students = {}
    with open(students_path, mode="r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            students[row["student_id"]] = row["name"]

    student_scores = defaultdict(list)
    seen_entries = set()  # Duplicate student-subject entries check karne ke liye

    # Step 2: Scores data read karna aur validate karna
    with open(scores_path, mode="r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            s_id, subject, marks = row["student_id"], row["subject"], float(row["marks"])

            # Validation: Unknown ID, Marks out of range (0-100), ya Duplicate entry reject karo
            if s_id not in students or not (0 <= marks <= 100) or (s_id, subject) in seen_entries:
                continue

            seen_entries.add((s_id, subject))
            student_scores[s_id].append(marks)

    report_data = []

    # Step 3: Har student ka average aur grade calculate karna
    for s_id, name in students.items():
        marks_list = student_scores[s_id]
        if not marks_list:
            continue

        avg = sum(marks_list) / len(marks_list)

        # Grading logic: A>=85, B>=70, C>=55, D>=40, warna F
        grade = (
            "A" if avg >= 85 
            else "B" if avg >= 70 
            else "C" if avg >= 55 
            else "D" if avg >= 40 
            else "F"
        )

        report_data.append({"student_id": s_id, "name": name, "average": round(avg, 2), "grade": grade})

    # Step 4: Sort karna (Average descending, phir Student ID ascending)
    report_data.sort(key=lambda x: (-x["average"], x["student_id"]))

    # Step 5: Ranks assign karna
    for rank, data in enumerate(report_data, 1):
        data["rank"] = rank

    # Step 6: Final report CSV file save karna
    with open(output_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["rank", "student_id", "name", "average", "grade"])
        writer.writeheader()
        writer.writerows(report_data)


# --- Sample Test ---
# Test files create karke check karna:
with open("students.csv", "w", newline="") as f:
    f.write("student_id,name\n101,Ali\n102,Sara\n")

with open("scores.csv", "w", newline="") as f:
    f.write("student_id,subject,marks\n101,Math,90\n101,Science,80\n101,English,85\n102,Math,70\n")

# Function run karna
generate_gradebook("students.csv", "scores.csv", "report.csv")
print("Gradebook report generated successfully!")

Gradebook report generated successfully!


# Task 12 |
.  Custom Iterable Range Toolkit
. Concepts: Iterators, generators, dunder methods
. Problem: Create a ReversibleRange(start,stop,step) class supporting iteration, reversed iteration, len(), and membership tests without constructing a list.
. Requirements: Match Python range stop-exclusive semantics for positive and negative nonzero steps. Implement __iter__, __reversed__, __len__, and __contains__.
. Sample / expected behavior: ReversibleRange(2,11,3) -> [2,5,8]; reversed -> [8,5,2]; 5 in object -> True.


In [16]:
class ReversibleRange:
    def __init__(self, start, stop, step=1):
        if step == 0:
            raise ValueError("Step zero nahi ho sakta!")
        self.start, self.stop, self.step = start, stop, step
        
        # Total elements calculate karne ka formula
        if (step > 0 and start < stop) or (step < 0 and start > stop):
            # Math.ceil((stop - start) / step) ka equivalent logic
            self._len = (abs(stop - start) + abs(step) - 1) // abs(step)
        else:
            self._len = 0

    # 1. Normal Iteration (__iter__)
    def __iter__(self):
        curr = self.start
        if self.step > 0:
            while curr < self.stop:
                yield curr
                curr += self.step
        else:
            while curr > self.stop:
                yield curr
                curr += self.step

    # 2. Reversed Iteration (__reversed__)
    def __reversed__(self):
        if self._len == 0:
            return iter([])
        # Last valid element se reverse chalna
        last_val = self.start + (self._len - 1) * self.step
        return ReversibleRange(last_val, self.start - self.step if self.step > 0 else self.start + abs(self.step), -self.step)

    # 3. Length check (__len__)
    def __len__(self):
        return self._len

    # 4. Membership test (__contains__) - O(1) time complexity
    def __contains__(self, val):
        if self._len == 0:
            return False
        # Check karo ke value range ke andar hai aur step ke mutabiq fit baithti hai
        if self.step > 0:
            return self.start <= val < self.stop and (val - self.start) % self.step == 0
        else:
            return self.stop < val <= self.start and (self.start - val) % abs(self.step) == 0

# --- Sample Test ---
r = ReversibleRange(2, 11, 3)

print("List:", list(r))              # Output: [2, 5, 8]
print("Reversed:", list(reversed(r))) # Output: [8, 5, 2]
print("Length:", len(r))              # Output: 3
print("Contains 5:", 5 in r)          # Output: True
print("Contains 4:", 4 in r)          # Output: False

List: [2, 5, 8]
Reversed: [8, 5, 2]
Length: 3
Contains 5: True
Contains 4: False


# Task 13 |
.  Booking Conflict Scheduler
. Concepts: Datetime, sorting, interval algorithms
. Problem: Implement a room booking scheduler. Each booking has room, start datetime, end datetime, and booking ID. Accept bookings .only when they do not overlap existing bookings for the same room.
. Requirements: Use datetime objects; reject end<=start and duplicate IDs. Adjacent bookings are allowed. Return conflicts for rejected requests.
. Sample / expected behavior: Room A: 09:00–10:00 and 10:00–11:00 accepted; 09:30–10:30 rejected.


In [17]:
from datetime import datetime


class RoomScheduler:

    def __init__(self):
        # Room-wise bookings store karne ke liye dictionary (Room -> List of Bookings)
        self.rooms = {}
        self.booking_ids = set()  # Duplicate IDs check karne ke liye

    def book(self, booking_id, room, start, end):
        # Step 1: Validation check (End time, Start time se pehle ya barabar na ho)
        if end <= start:
            raise ValueError("End time start time ke baad honi chahiye")

        # Duplicate ID check
        if booking_id in self.booking_ids:
            raise ValueError("Booking ID pehle se mojood hai")

        # Agar room pehli baar aaya hai toh list initialize karo
        if room not in self.rooms:
            self.rooms[room] = []

        # Step 2: Overlap check karna (Existing bookings ke sath)
        for b_id, b_start, b_end in self.rooms[room]:
            # Overlap condition: (Start1 < End2) aur (Start2 < End1)
            # (Note: Adjacent bookings allow hain, isliye strict < use kiya hai)
            if start < b_end and b_start < end:
                return {
                    "status": "Rejected",
                    "reason": f"Conflict with booking {b_id}",
                }

        # Step 3: Booking accept karo aur list mein store karo
        self.rooms[room].append((booking_id, start, end))
        self.booking_ids.add(booking_id)
        return {"status": "Accepted", "booking_id": booking_id}


# --- Sample Test ---
scheduler = RoomScheduler()

# Datetime objects banana
s1 = datetime(2026, 11, 7, 9, 0)
e1 = datetime(2026, 11, 7, 10, 0)

s2 = datetime(2026, 11, 7, 10, 0)  # Adjacent (Allowed)
e2 = datetime(2026, 11, 7, 11, 0)

s3 = datetime(2026, 11, 7, 9, 30)  # Overlapping (Rejected)
e3 = datetime(2026, 11, 7, 10, 30)

print(scheduler.book("B1", "Room A", s1, e1))  # Accepted
print(scheduler.book("B2", "Room A", s2, e2))  # Accepted (Adjacent)
print(scheduler.book("B3", "Room A", s3, e3))  # Rejected (Conflict)

{'status': 'Accepted', 'booking_id': 'B1'}
{'status': 'Accepted', 'booking_id': 'B2'}
{'status': 'Rejected', 'reason': 'Conflict with booking B1'}


# Task 14 |
.  Command-Line Contact Manager
. Concepts: Modules, JSON, CRUD, validation
. Problem: Build a menu-driven CLI contact manager with add, search, update, delete, list, and persistent JSON storage. Each contact has a unique ID, name, phone, and email.
. Requirements: Use separate functions/modules for storage and business logic. Handle missing/corrupt JSON safely without silently deleting existing data. Search names case-insensitively.
. Sample / expected behavior: Add ID=1, name="Ayesha"; restart application; search "aye" -> contact remains available.


In [18]:
import json
import os


class ContactManager:

    def __init__(self, filepath="contacts.json"):
        self.filepath = filepath
        self.contacts = self._load()

    # Storage: Safe JSON loading (Missing ya corrupt file ko safely handle karna)
    def _load(self):
        if not os.path.exists(self.filepath):
            return {}
        try:
            with open(self.filepath, "r", encoding="utf-8") as f:
                return json.load(f)
        except json.JSONDecodeError:
            # File corrupt ho toh data delete nahi hoga, safety ke liye empty dict return hogi
            print("Warning: Contacts file corrupt hai!")
            return {}

    # Storage: Data ko JSON file mein save karna
    def _save(self):
        with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump(self.contacts, f, indent=4)

    # 1. Add Contact (Unique ID ke sath naya contact banana)
    def add(self, cid, name, phone, email):
        if cid in self.contacts:
            raise ValueError("ID pehle se mojood hai!")
        self.contacts[cid] = {"name": name, "phone": phone, "email": email}
        self._save()

    # 2. Search Contact (Case-insensitive name search)
    def search(self, query):
        q = query.lower()
        return {
            cid: c for cid, c in self.contacts.items() if q in c["name"].lower()
        }

    # 3. Update Contact
    def update(self, cid, name=None, phone=None, email=None):
        if cid not in self.contacts:
            raise ValueError("Contact nahi mila!")
        if name:
            self.contacts[cid]["name"] = name
        if phone:
            self.contacts[cid]["phone"] = phone
        if email:
            self.contacts[cid]["email"] = email
        self._save()

    # 4. Delete Contact
    def delete(self, cid):
        if cid in self.contacts:
            del self.contacts[cid]
            self._save()

    # 5. List All Contacts
    def list_all(self):
        return self.contacts


# --- Sample Test & Persistence Simulation ---
filename = "test_contacts.json"

# Pehli baar contact add karna
cm = ContactManager(filename)
cm.add("1", "Ayesha", "03001234567", "ayesha@mail.com")

# Application restart simulation (Naya instance file se data dobara load karega)
cm_restarted = ContactManager(filename)
print(
    "Search 'aye':", cm_restarted.search("aye")
)  # Output: {'1': {'name': 'Ayesha', ...}}

Search 'aye': {'1': {'name': 'Ayesha', 'phone': '03001234567', 'email': 'ayesha@mail.com'}}


# Task 15 |
. Integrated Order Fulfillment System
. Concepts: OOP, composition, files, exceptions, algorithms
. Problem: Design Product, Cart, Order, and Inventory classes. Read a product catalog from JSON, build carts, apply percentage discounts, compute totals, and place orders with stock updates. Persist orders to JSON.
. Requirements: Reject invalid quantities and out-of-stock purchases. Validate all items before changing inventory; roll back if order persistence fails. Generate unique order IDs. Include at least three unit tests using unittest.
. Sample / expected behavior: Catalog: P1 price=100 stock=5; cart P1 x2 with 10% discount -> total 180, stock becomes 3 after successful order.

In [19]:
import json
import os
import unittest
import uuid


class Product:

    def __init__(self, id, name, price, stock):
        self.id, self.name, self.price, self.stock = (
            id,
            name,
            float(price),
            int(stock),
        )


class Inventory:

    def __init__(self, filepath="catalog.json"):
        self.filepath = filepath
        self.products = {}
        self.load()

    # Step 1: JSON file se product catalog load karna
    def load(self):
        if os.path.exists(self.filepath):
            with open(self.filepath, "r", encoding="utf-8") as f:
                for p in json.load(f):
                    self.products[p["id"]] = Product(**p)


class Cart:

    def __init__(self):
        self.items = {}  # {product_id: quantity}

    def add(self, product_id, qty):
        if qty <= 0:
            raise ValueError("Quantity positive honi chahiye")
        self.items[product_id] = self.items.get(product_id, 0) + qty

    # Step 2: Total cost calculate karna with optional percentage discount
    def calculate_total(self, inventory, discount_pct=0):
        total = 0
        for pid, qty in self.items.items():
            if pid not in inventory.products:
                raise ValueError(f"Product {pid} mojood nahi hai")
            total += inventory.products[pid].price * qty
        return total * (1 - discount_pct / 100)


class OrderSystem:

    def __init__(self, inventory, orders_file="orders.json"):
        self.inventory = inventory
        self.orders_file = orders_file

    def place_order(self, cart, discount_pct=0):
        # Step 3: Stock validation BEFORE changing any inventory
        for pid, qty in cart.items.items():
            if (
                pid not in self.inventory.products
                or self.inventory.products[pid].stock < qty
            ):
                raise ValueError(f"Out of stock ya invalid product: {pid}")

        # Step 4: Unique Order ID generate karna aur total calculate karna
        order_id = str(uuid.uuid4())[:8]
        total = cart.calculate_total(self.inventory, discount_pct)

        # Step 5: Temporary stock deduction & rollback tracking
        stock_backups = {}
        try:
            for pid, qty in cart.items.items():
                stock_backups[pid] = self.inventory.products[pid].stock
                self.inventory.products[pid].stock -= qty

            # Step 6: Orders ko JSON file mein persist (save) karna
            order_data = {
                "order_id": order_id,
                "items": cart.items,
                "total": total,
            }
            orders = []
            if os.path.exists(self.orders_file):
                with open(self.orders_file, "r", encoding="utf-8") as f:
                    orders = json.load(f)
            orders.append(order_data)

            with open(self.orders_file, "w", encoding="utf-8") as f:
                json.dump(orders, f, indent=4)

        except Exception as e:
            # Rollback: Agar file save mein error aaye toh stock wapas restore karo
            for pid, old_stock in stock_backups.items():
                self.inventory.products[pid].stock = old_stock
                raise RuntimeError(f"Order failed, rolled back: {e}")

        return order_id


# --- Unit Tests using unittest ---
class TestOrderSystem(unittest.TestCase):

    def setUp(self):
        # Test files create karna
        self.catalog_path = "test_catalog.json"
        self.orders_path = "test_orders.json"
        catalog_data = [{"id": "P1", "name": "Laptop", "price": 100, "stock": 5}]
        with open(self.catalog_path, "w", encoding="utf-8") as f:
            json.dump(catalog_data, f)

        self.inv = Inventory(self.catalog_path)
        self.system = OrderSystem(self.inv, self.orders_path)

    def tearDown(self):
        # Test ke baad files delete karna
        for p in [self.catalog_path, self.orders_path]:
            if os.path.exists(p):
                os.remove(p)

    def test_1_successful_order(self):
        # Test 1: Successful order, 10% discount & stock update
        cart = Cart()
        cart.add("P1", 2)
        total = cart.calculate_total(self.inv, discount_pct=10)
        self.assertEqual(total, 180.0)  # (200 - 10%) = 180.0

        order_id = self.system.place_order(cart, discount_pct=10)
        self.assertTrue(order_id)
        self.assertEqual(
            self.inv.products["P1"].stock, 3
        )  # Stock 5 se 3 reh gaya

    def test_2_out_of_stock(self):
        # Test 2: Out of stock purchase reject honi chahiye
        cart = Cart()
        cart.add("P1", 10)  # Stock sirf 5 hai
        with self.assertRaises(ValueError):
            self.system.place_order(cart)

    def test_3_invalid_quantity(self):
        # Test 3: Zero ya negative quantity reject honi chahiye
        cart = Cart()
        with self.assertRaises(ValueError):
            cart.add("P1", 0)


# --- Run Tests ---
if __name__ == "__main__":
    unittest.main(argv=[""], exit=False)

...
----------------------------------------------------------------------
Ran 3 tests in 0.070s

OK
